In [1]:
source(here::here('MG3.functions1.R'))

# Writing species into the database

In [2]:
assembyfilename=paste0(StaticInpDir,"assemblies.json")
assembltext=readLines(assembyfilename)
assembliesJSON <- fromJSON(assembltext)

In [3]:
GenomesDF=NULL
for (assembly in assembliesJSON$reports ){
    row=list(accession=assembly$accession, sample= assembly$assembly_info$biosample$accession,
            taxon= assembly$assembly_info$biosample$description$organism$tax_id)
    if (is.null(row$taxon)){
        row$taxon=assembly$organism$tax_id
    }
    if (is.null(row$sample)){
        row$sample=NA
    }
    if (is.null(GenomesDF)){
        GenomesDF=as.data.frame(row)
    }else{
        GenomesDF=rbind(GenomesDF, as.data.frame(row))
    }
}


In [4]:
# run ONCE

if ("genomes" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE genomes')
dbWriteTable(mydbRW, "genomes", GenomesDF, append=FALSE)

dbExecute(mydbRW, 'CREATE INDEX GenomesAccessionInd ON genomes(accession)')
dbExecute(mydbRW, 'CREATE INDEX GenomesTaxonInd ON genomes(taxon)')
dbExecute(mydbRW, 'CREATE INDEX GenomesSampleInd ON genomes(sample)')


[1] 0

[1] 0

[1] 0

# Saving chromosome lengths to SQLite

In [5]:
# run ONCE

if ("seqinfo" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE seqinfo')

for (rowNumber in 1:nrow(GenomesDF)){
    if (rowNumber %% 20 == 0){
        message(rowNumber, "/", nrow(GenomesDF))
    }
    accession=GenomesDF$accession[rowNumber]
    #message(accession)
    chrsDF=getGenomeChrs(accession)
    #ID which CHR name format is used
    genomeDir=paste0(StaticInpDir, accession,"/")
    #message(genomeDir)
    fs=list.files(genomeDir, pattern = paste0(accession,".*_genomic.fna$"), full.names = TRUE)
    if (length(fs)!=1){
        message(accession)
        message(paste(accession, "PROBLEM"))
        break
    }
    firstLine=readLines(con = fs[1],n = 1)
    #message(firstLine)
    molname=str_extract(string = firstLine, pattern = "^>\\s*([^\\s]+)\\s+.*$", group = 1)
    #message(molname)
    colfound=FALSE
    for (cn in colnames(chrsDF)){
        if ( molname %in% chrsDF[[cn]]){
            #message(cn)
            colfound=TRUE
            break
        }
    }
    if (! colfound){
        message(accession)
        message(paste(accession, "COLUMN PROBLEM"))
        break
    }else{
       colnames(chrsDF)[colnames(chrsDF) == cn]="seqnames"
       colnames(chrsDF)[colnames(chrsDF) == "assemblyAccession"]="accession"
       colstoexport=c("accession", "seqnames", "chrName", "length")
       chrsDF$length=as.integer(chrsDF$length)
       dbWriteTable(mydbRW, "seqinfo", chrsDF[,colstoexport], append=TRUE)
        
    }
}

dbExecute(mydbRW, 'CREATE INDEX seqinfoAccessionSeqnamesInd ON seqinfo(accession,seqnames)')
dbExecute(mydbRW, 'CREATE INDEX seqinfoSeqnamesInd ON seqinfo(seqnames)')

20/223

40/223

60/223

80/223

100/223

120/223

140/223

160/223

180/223

200/223

220/223



[1] 0

[1] 0

In [6]:
length(unique(GenomesDF$taxon))

[1] 160

# Saving the GTFs

In [8]:
# Run ONCE
system(paste0("mkdir ", OutputDir,"/tmpGTF2CSV/slurm_logs/ -p" ), wait = TRUE) #creating the dir
if ("GTFs" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE GTFs')
swarmparams=paste0("#SWARM -t 5 -g 10 --time=10:00:00 --gres=lscratch:10 --logdir ", fs::path_expand(paste0(OutputDir,"/tmpGTF2CSV/slurm_logs/")))
scriptname="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh"
swarmlines=c(swarmparams)

for (rowNumber in 1:nrow(GenomesDF)){
    accession = GenomesDF$accession[rowNumber]
    tmpline=paste(scriptname, "~/data/lambda/notebooks/importGTFsforStep02.r", accession)
    swarmlines=addToVec(swarmlines, tmpline)
}
fname=paste0(OutputDir,"tmpGTF2CSV/importGTFsToSQL-", getTimeStamp(),".swarm")
writeLines(text = swarmlines, con=fname)
message("resulting file: ", fname, " \n run it from bash")

resulting file: ~/data/lambda/output/tmpGTF2CSV/importGTFsToSQL-2026-01-13_18.16.32.605372.swarm 
 run it from bash



In [33]:
#run ONCE
# importing CSVs to SQLite
if ("GTFs" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE GTFs')
dbExecute(mydbRW, 'CREATE TABLE GTFs("bsid" TEXT, "seqnames" TEXT, "start" UNSIGNED BIG INT, "end" UNSIGNED BIG INT, "width" UNSIGNED BIG INT, "strand" TEXT, "source" TEXT, "type" TEXT, "gene" TEXT, "protein_id" TEXT, "Name" TEXT, "accession" TEXT)')
fs=list.files(paste0(OutputDir,"tmpGTF2CSV"), pattern = "*.csv$", full.names = TRUE)
for (f in fs){
    comToBash=sprintf('sqlite3 %s/%s -cmd ".mode csv" ".import %s GTFs" ".quit"', OutputDir, SQLfname,
                     fs::path_expand(f) )
    #message(comToBash)
    retval=system(comToBash, intern = TRUE, wait = TRUE)

}

[1] 0

In [34]:
# Run ONCE
dbExecute(mydbRW, "ALTER TABLE GTFs DROP COLUMN bsid")
dbExecute(mydbRW, "DELETE FROM GTFs WHERE GTFs.accession='accession'")
message("Cleanup")
dbExecute(mydbRW, 'CREATE INDEX GTFsAccGeneInd ON GTFs(accession,gene)')
message("Ind1")
dbExecute(mydbRW, 'CREATE INDEX GTFsMainCoordInd ON GTFs(accession,seqnames,start,end,strand)')
message("Ind2")
dbExecute(mydbRW, 'CREATE INDEX GTFsEndCoordInd ON GTFs(accession,seqnames,end,strand)')
message("Ind3")
dbExecute(mydbRW, 'CREATE INDEX GTFsNameInd ON GTFs(Name)')
message("Ind4")
dbExecute(mydbRW, 'CREATE INDEX GTFsAccNameInd ON GTFs(accession,Name)')
message("Ind5")
dbExecute(mydbRW, 'CREATE INDEX GTFsgeneInd ON GTFs(gene)')
message("Ind6")
dbExecute(mydbRW, 'CREATE INDEX GTFsAccProtInd ON GTFs(accession,protein_id)')
message("In7")
dbExecute(mydbRW, 'CREATE INDEX GTFsProtInd ON GTFs(protein_id)')
message("Ind8")
system("echo 1 > ~/data/lambda/output/doneGTFs", intern = TRUE, wait = TRUE)

[1] 0

[1] 183

Cleanup



[1] 0

Ind1



...

[1] 0

In7



[1] 0

Ind8



character(0)

# Saving taxonomy to SQL

In [17]:
#run ONCE
if ("taxonomy" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE taxonomy')

taxonomyList=classification(GenomesDF$taxon, db = 'ncbi')
taxonomyList2=lapply(X = taxonomyList, FUN=function(X){X$orderTax=1:nrow(X)
                                                      return(X)})
taxonomyDF=NULL
for ( i in 1:length(taxonomyList2)){
    accession=GenomesDF$accession[i]
    tmpDF=taxonomyList2[[i]]
    tmpDF$accession=accession
    taxonomyDF=addToDF(taxonomyDF, tmpDF)
}
dbWriteTable(mydbRW, "taxonomy", taxonomyDF, append=FALSE)

dbExecute(mydbRW, 'CREATE INDEX TaxonomyIdInd ON taxonomy(id)')
dbExecute(mydbRW, 'CREATE INDEX TaxonomyAccessionInd ON taxonomy(accession)')
dbExecute(mydbRW, 'CREATE INDEX TaxonomyAccessionOrderInd ON taxonomy(accession,orderTax)')

[1] 0

[1] 0

[1] 0

# Running RMSK

In [18]:
getGenomeChrs=function(inAccession){
    inFiname=paste0(StaticInpDir, inAccession, "/sequence_report.jsonl")
    inLines=readLines(inFiname)
    chrlens=as.integer(str_extract(string = inLines, pattern = ',\\s*"length"\\s*:\\s*(\\d+)\\s*,',group = 1 ))
    boolmask=(chrlens>=minCHrLim)
    boolmask[is.na(boolmask)]=FALSE
    goodLines=inLines[boolmask]
    thetext=paste0(goodLines, collapse = ",\n")

    thetext=paste0('{"assemblies": [',thetext,"]}")
    result <- fromJSON(thetext)
    outDF=NULL
    for (assembly in result$assemblies){
        if (is.null(outDF)){
            outDF=as.data.frame(assembly)
        }else{
            extracols=setdiff(colnames(outDF), names(assembly))
            for (cn in extracols){
                assembly[[cn]]=NA
            }
            extracols=setdiff(names(assembly), colnames(outDF))
            for (cn in extracols){
                outDF[[cn]]=NA
            }
            outDF=rbind(outDF, as.data.frame(assembly))
        }
    }
    return(outDF)
}



In [22]:
system(paste0("mkdir ", OutputDir,"/RMSK/slurm_logs/ -p" ), wait = TRUE) #creating the dir

sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R"
swarmlines=NULL #--time 10:00:00 --gres=lscratch:100 --mem=20g --cpus-per-task=5
for (rowNumber in 1:nrow(GenomesDF)){
    accession=GenomesDF$accession[rowNumber]
    
    chrsDF=getGenomeChrs(accession)
    #ID which CHR name format is used
    genomeDir=paste0(StaticInpDir, accession,"/")
    #message(genomeDir)
    fs=list.files(genomeDir, pattern = paste0(accession,".*_genomic.fna$"), full.names = TRUE)
    if (length(fs)!=1){
        message(accession)
        message(paste(accession, "PROBLEM"))
        break
    }
    firstLine=readLines(con = fs[1],n = 1)
    #message(firstLine)
    molname=str_extract(string = firstLine, pattern = "^>\\s*([^\\s]+)\\s+.*$", group = 1)
    #message(molname)
    colfound=FALSE
    for (cn in colnames(chrsDF)){
        if ( molname %in% chrsDF[[cn]]){
            #message(cn)
            colfound=TRUE
            break
        }
    }
    if (! colfound){
        message(accession)
        message(paste(accession, "COLUMN PROBLEM"))
        break
    }else{
        #message(cn)
        #add the sbatch call here
        #taxonID=getWorkingTaxonID(accession)#GenomesDF$taxon[rowNumber]
        for (chrname in chrsDF[[cn]]){
            #checking if 
            #comToBash=paste0("sbatch --time 10:00:00 --gres=lscratch:100 --mem=20g --cpus-per-task=5  ",sbatchscript," ",accession," ",chrname," ",taxonID," ~/data/lambda/multigene2/Genomes/ ~/data/lambda/multigene2/RMSK/")
            #system(comToBash, intern = FALSE, wait = FALSE)
            swarmline=paste0(sbatchscript," ",accession," ",chrname," ", OutputDir, "/RMSK/")
            swarmlines=addToVec(swarmlines, swarmline)
        }
    }
}



In [23]:
swarmlines[1:2]

[1] "~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R GCF_027475565.1 NC_069380.1 ~/data/lambda/output//RMSK/"
[2] "~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R GCF_027475565.1 NC_069382.1 ~/data/lambda/output//RMSK/"

In [27]:
#run ONCE
swarmparams=paste0("#SWARM -t 5 -g 20 --time=10:00:00 --gres=lscratch:100 --logdir ", path.expand(OutputDir),"/RMSK/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"/RMSK/runningRMSKperCHR.",i,"-", getTimeStamp(),".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    swarmFiles=addToVec(swarmFiles, fname)
    #message("Run from bash: ", fname)
    processed=upperlimit
    i=i+1
}


1 1000

1001 2000

2001 3000

3001 4000

4001 5000

5001 6000

6001 7000

7001 7693



In [ ]:
#Run once
startedBool=rep(FALSE, length(swarmFiles))
while (!all(startedBool)){
    for (i in 1:length(swarmFiles)){
        if (!startedBool[i]){
            comToBash=paste0("swarm ", swarmFiles[i])
            message(comToBash)
            retval=system(comToBash, wait = TRUE)
            if (retval == 0){
                startedBool[i]=TRUE
                message("\t Success")
            }else{
                message("\t Nope")
                Sys.sleep(2)
            }
        }
    }
    if (!all(startedBool)){
        message(sum(!startedBool) ,"left")
        Sys.sleep(120)
    }
}
system('echo 1 > /home/frimanav/data/lambda/output/doneRMSKschedulting', intern = TRUE, wait = TRUE)


swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.0-2026-01-13_18.48.56.239094.swarm

	 Success

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.1-2026-01-13_18.48.56.248132.swarm

	 Success

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.2-2026-01-13_18.48.56.284561.swarm

	 Success

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.3-2026-01-13_18.48.56.303986.swarm

	 Nope

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.4-2026-01-13_18.48.56.325485.swarm

	 Nope

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.5-2026-01-13_18.48.56.332126.swarm

	 Nope

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.6-2026-01-13_18.48.56.340563.swarm

	 Nope

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR.7-2026-01-13_18.48.56.349169.swarm

	 Success

4left



## Checking if the work is done

In [13]:
# Checking completion of the repeatmasker run
minBEDfileSize=5000
problemChrsDF=NULL
for (rowNumber in 1:nrow(GenomesDF)){
    accession=GenomesDF$accession[rowNumber]
    if (rowNumber %% 10 == 0) message(rowNumber, '/', nrow(GenomesDF))
    chrsDF=getGenomeChrs(accession)
    #ID which CHR name format is used
    genomeDir=paste0(StaticInpDir, accession,"/")
    #message(genomeDir)
    fs=list.files(genomeDir, pattern = paste0(accession,".*_genomic.fna"), full.names = TRUE)
    
    if (length(fs)<1){
        message(accession)
        message(paste(accession, "PROBLEM"))
        break
    }
    largest_fasta_size=-1
    full_fasta_path=NULL
    for (fname in fs){
        if (file.info(fname)$size > largest_fasta_size){
            largest_fasta_size=file.info(fname)$size
            full_fasta_path=fname
        }
    }
    firstLine=readLines(con = full_fasta_path,n = 1)
    #message(firstLine)
    molname=str_extract(string = firstLine, pattern = "^>\\s*([^\\s]+)\\s+.*$", group = 1)
    #message(molname)
    colfound=FALSE
    for (cn in colnames(chrsDF)){
        if ( molname %in% chrsDF[[cn]]){
            #message(cn)
            colfound=TRUE
            break
        }
    }
    if (! colfound){
        message(accession)
        message(paste(accession, "COLUMN PROBLEM"))
        break
    }else{
        #message(cn)
        #Check if the bed file exists
        assemblyDir=paste0(StaticInpDir,accession)
        RMSKDir=paste0(OutputDir,"/RMSK/",accession)
        for (k in 1:length(chrsDF[[cn]])){
            chrname=chrsDF[[cn]][k]
            chrlen=chrsDF$length[k]
            if (chrlen < minCHrLim) next
            assemblyFiles=list.files(RMSKDir, pattern = paste0(accession,".*", chrname,".fna.repeatMasker.bed"), full.names = TRUE)
            if (length(assemblyFiles)!=1){
                tmpDF=data.frame(accession=accession, chr=chrname, chrlen=chrlen, nfiles=length(assemblyFiles))
                problemChrsDF=addToDF(problemChrsDF, tmpDF)
            }else{
                if (file.info(assemblyFiles)$size < minBEDfileSize){
                    tmpDF=data.frame(accession=accession, chr=chrname, chrlen=chrlen, nfiles=length(assemblyFiles), size=file.info(assemblyFiles)$size, numchrs=length(chrsDF[[cn]]))
                    problemChrsDF=addToDF(problemChrsDF, tmpDF)
                }
            }
        }
        #GCF_028885625.2_NHGRI_mPonPyg2-v2.0_pri_genomic.fna_NC_072395.2.fna.repeatMasker.bed
        
        
    }
    
}

10/223

20/223

30/223

40/223

50/223

60/223

70/223

80/223

90/223

100/223

110/223

120/223

130/223

140/223

150/223

160/223

170/223

180/223

190/223

200/223

210/223

220/223



In [14]:
problemChrsDF = as.data.frame( problemChrsDF %>% group_by(accession) %>% mutate(numproblem=n()))

In [25]:
problemChrsDF=problemChrsDF[ problemChrsDF$chrlen>lookaroundDist & is.na(problemChrsDF$size), ]

In [26]:
problemChrsDF

,accession,chr,chrlen,nfiles,size,numchrs,numproblem
,<chr>,<chr>,<dbl>,<int>,<dbl>,<int>,<int>
24,GCF_000003205.7,NC_007305.6,113155487,0,NA,NA,1
25,GCF_023091745.1,NC_068910.1,221681955,0,NA,NA,14
40,GCF_036321535.1,NC_087449.1,70542807,0,NA,NA,10
102,GCF_033118175.1,NC_083710.1,90020548,0,NA,NA,48
150,GCF_030445035.2,NC_080697.1,65148191,0,NA,NA,1
183,GCF_029281585.2,NC_073244.2,80312928,0,NA,NA,1
184,GCF_964656455.1,NC_135721.1,139080281,0,NA,NA,1
204,GCA_000230815.1,CM001276.1,232296185,0,NA,NA,1
205,GCF_049350105.2,NC_133406.1,240198992,0,NA,NA,1


### Trying to ID problem taxa (not needed. do not run)

In [27]:
system(paste0("mkdir ", OutputDir,"/RMSK/slurm_logs2/ -p" ), wait = TRUE) #creating the dir

sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R"
swarmlines2=NULL #--time 10:00:00 --gres=lscratch:100 --mem=20g --cpus-per-task=5
for (i in 1:nrow(problemChrsDF)){
    accession=problemChrsDF$accession[i]
    chrname=problemChrsDF$chr[i]
    swarmline=paste0(sbatchscript," ",accession," ",chrname," ", OutputDir, "/RMSK/")
    swarmlines2=addToVec(swarmlines2, swarmline)
}


In [28]:
length(swarmlines2)

[1] 24

In [32]:
swarmlines2[1:2]

[1] "~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R GCF_000003205.7 NC_007305.6 ~/data/lambda/output//RMSK/"
[2] "~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R GCF_023091745.1 NC_068910.1 ~/data/lambda/output//RMSK/"

In [30]:
#run ONCE
swarmparams=paste0("#SWARM -t 5 -g 30 --time=24:00:00 --gres=lscratch:150 --logdir ", path.expand(OutputDir),"/RMSK/slurm_logs2/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines2)){
    upperlimit=processed+min(maxchunk, length(swarmlines2)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"/RMSK/runningRMSKperCHR_attempt2-.",i,"-", getTimeStamp(),".swarm")
    swarmFiles=addToVec(swarmFiles, fname)
    writeLines(text = c(swarmparams, swarmlines2[(processed+1):upperlimit]), 
               con=fname)
    #message("Run from bash: ", fname)
    processed=upperlimit
    i=i+1
}

1 24



In [31]:
startedBool=rep(FALSE, length(swarmFiles))
while (!all(startedBool)){
    for (i in 1:length(swarmFiles)){
        if (!startedBool[i]){
            comToBash=paste0("swarm ", swarmFiles[i])
            message(comToBash)
            retval=system(comToBash, wait = TRUE)
            if (retval == 0){
                startedBool[i]=TRUE
                message("\t Success")
            }else{
                message("\t Nope")
                Sys.sleep(2)
            }
        }
    }
    if (!all(startedBool)){
        message(sum(!startedBool) ,"left")
        Sys.sleep(120)
    }
}
system('echo 1 > /home/frimanav/data/lambda/output/doneRMSKschedulting-attempt2', intern = TRUE, wait = TRUE)

swarm ~/data/lambda/output//RMSK/runningRMSKperCHR_attempt2-.0-2026-01-14_12.48.57.365692.swarm

	 Success



character(0)

## Saving RMSK bed files to SQL

In [ ]:
#run ONCE


queryTxt='SELECT DISTINCT accession FROM seqinfo'
accessionsTMP=dbGetQuery(mydbRW, queryTxt)[["accession"]]
combinedRMSKdfList=list()
for (i in 1:length(accessionsTMP)){
    accession = accessionsTMP[i]
    if (i %% 10 == 0) message(paste(i,"/",length(accessionsTMP)))
    assemblyDir=paste0(OutputDir,"RMSK/",accession,"/")
    queryTxt=sprintf('SELECT DISTINCT seqnames FROM seqinfo WHERE accession="%s"', accession)
    chrNames=dbGetQuery(mydbRO, queryTxt)[["seqnames"]]
    for (chrname in chrNames){
        assemblyFiles=list.files(assemblyDir, pattern = paste0(accession,".*", chrname,".fna.repeatMasker.bed"), full.names = TRUE)
        if (length(assemblyFiles)==1){
            if (file.info(assemblyFiles)$size >= minBEDfileSize){
                tmpDF=read.table(file = assemblyFiles, sep = "\t", col.names = c("seqnames", "start", "end", "descr", "score", "strand"))
                tmpDF$score=NULL
                tmpCols=c("seqnames", "start", "end", "strand", "descr")
                tmpDF2=tmpDF[, tmpCols]
                tmpDF2$accession=accession
                tmpDF2$start=tmpDF2$start +1 #fixing the zero-based BED format
                tmpDF2=tmpDF2[tmpDF2$end - tmpDF2$start >= minRMSKinsertionlen,]
                if (nrow(tmpDF2)>0){
                    combinedRMSKdfList[[length(combinedRMSKdfList)+1]]=tmpDF2
                }
            }
        }else{
            message("Problem ", accession," ", chrname)
        }
    }
}

combinedRMSKdf=bind_rows(combinedRMSKdfList)

if ("RMSK" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE RMSK')
dbWriteTable(mydbRW, "RMSK", combinedRMSKdf, append=FALSE)
message("Start indexing")
dbExecute(mydbRW, 'CREATE INDEX RMSKaccessionInd ON RMSK(accession)')
message("Ind 1")
dbExecute(mydbRW, 'CREATE INDEX RMSKaccessionSeqnamesInd ON RMSK(accession, seqnames)')
message("Ind 2")
dbExecute(mydbRW, 'CREATE INDEX RMSKSeqnamesInd ON RMSK(seqnames)')
message("Ind 3")
dbExecute(mydbRW, 'CREATE INDEX RMSKCoordInd ON RMSK(seqnames, start, end, strand)')
message("Ind 4")
dbExecute(mydbRW, 'CREATE INDEX RMSKAccessionCoordInd ON RMSK(accession, seqnames, start, end, strand)')
message("Ind 5")
system('echo 1 > /home/frimanav/data/lambda/output/doneRMSKsaving', intern = TRUE, wait = TRUE)


10 / 223



# check large chromosomes have RMSK

In [4]:
queryTxt=sprintf('SELECT * FROM seqinfo WHERE length > %d', 5e6)
longCHRsDF=dbGetQuery(mydbRO, queryTxt)


In [5]:
problemChrsDF2=NULL
for (accession in unique(longCHRsDF$accession)){
    subsetDF=longCHRsDF[ longCHRsDF$accession == accession, ]
    longchrnames=subsetDF$seqnames
    queryTxt=sprintf('SELECT DISTINCT seqnames FROM RMSK WHERE accession=="%s"', accession)
    rmskCHRsDFtmp=dbGetQuery(mydbRO, queryTxt)
    if ( !all(longchrnames %in% rmskCHRsDFtmp$seqnames)){
        message("Not all chrs of ", accession, " . Num found: ", sum(longchrnames %in% rmskCHRsDFtmp$seqnames),"/",length(longchrnames ))
        longestchr=max(subsetDF$length[! longchrnames %in% rmskCHRsDFtmp$seqnames])
        namelongest=subsetDF$chrName[ subsetDF$length==longestchr]
        message("\tLongest problem: ",longestchr)
        tmpDF=data.frame(accession=accession, chr=longchrnames[! longchrnames %in% rmskCHRsDFtmp$seqnames])
        problemChrsDF2=addToDF(problemChrsDF2, tmpDF)

    }
}



Not all chrs of GCF_036321535.1 . Num found: 43/44

	Longest problem: 5577944

Not all chrs of GCF_047675955.1 . Num found: 30/33

	Longest problem: 6591012

Not all chrs of GCF_033118175.1 . Num found: 54/55

	Longest problem: 5629527

Not all chrs of GCF_020826845.1 . Num found: 51/52

	Longest problem: 8065355

Not all chrs of GCF_028021215.1 . Num found: 24/25

	Longest problem: 12442647

Not all chrs of GCA_023701675.1 . Num found: 31/32

	Longest problem: 5592578

Not all chrs of GCF_028885655.2 . Num found: 24/25

	Longest problem: 212916775



In [6]:
problemChrsDF2

accession,chr
<chr>,<chr>
GCF_036321535.1,NW_026989811.1
GCF_047675955.1,NW_027326745.1
GCF_047675955.1,NW_027327481.1
GCF_047675955.1,NW_027327592.1
GCF_033118175.1,NW_026870885.1
GCF_020826845.1,NW_026690854.1
GCF_028021215.1,NW_027175249.1
GCA_023701675.1,JALAIW010000029.1
GCF_028885655.2,NC_071988.2


In [ ]:
sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/prepareChrFastaAndRun_NCBIrepeatMaskerChromosome.sh.R"
swarmlines3=NULL 
for (rowNumber in 1:nrow(problemChrsDF2)){
    accession=problemChrsDF2$accession[rowNumber]
    chrname=problemChrsDF2$chr[rowNumber]
    #ID which CHR name format is used
    genomeDir=paste0(StaticInpDir, accession,"/")
    workingTaxon=NULL
    tmpTaxDF=getTaxonID(accession)  
    for (j in 1:max(5,nrow(tmpTaxDF))){
        taxIDtmp=tmpTaxDF$id[j]
        if (testRepeatMaskerDB(taxIDtmp)){
            workingTaxon=taxIDtmp
            break
        }
    }
    if (is.null(workingTaxon)){
        message("Cannot find working taxon: ", accession)
    }else{
        swarmline=paste0(sbatchscript," ",accession," ",chrname," ",workingTaxon, " ", OutputDir, "/RMSK/")
        swarmlines3=addToVec(swarmlines3, swarmline)
    }
}

In [ ]:
#run ONCE
swarmparams=paste0("#SWARM -t 10 -g 200 --time=48:00:00 --gres=lscratch:500 --logdir ", path.expand(OutputDir),"/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines3)){
    upperlimit=processed+min(maxchunk, length(swarmlines3)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"/RMSK/runningRMSKperCHR_attempt3-.",i,"-", getTimeStamp(),".swarm")
    swarmFiles=addToVec(swarmFiles, fname)
    writeLines(text = c(swarmparams, swarmlines3[(processed+1):upperlimit]), 
               con=fname)
    message("Run from bash: ", fname)
    processed=upperlimit
    i=i+1
}

In [ ]:
tmpTaxDF

# Making a list of TE coding genes wrongfully annotated in the host genomes

In [35]:
# run ONCE
i=0
tmpDir=paste0(OutputDir,"filteroutTEgenes/")
system(paste0("mkdir ", tmpDir, " -p"), wait=TRUE)
for (accession in GenomesDF$accession){
    if (i %% 10 == 0) message(i, " / ", length(GenomesDF$accession))
    proteinFA=paste0(StaticInpDir,"/", accession, "/protein.faa")    
    jobName=sprintf("InterPro.%s", accession)
    comToBash=sprintf("ml interproscan/5.72-103.0; cd %s ; interproscan --applications PANTHER,Pfam %s %s 1000; cd %s/%s ; sbatch interproscan.batch", tmpDir, proteinFA, jobName, tmpDir,jobName)
    queued_success=FALSE
    while (!queued_success){
        retval=system(comToBash, intern = FALSE, wait = TRUE)
        if (retval == 0){
            queued_success=TRUE
        }else{
            message("Sleeping i=",i," accession=",accession)
            Sys.sleep(60)
        }
    }
    
    i=i+1
}

0 / 223

10 / 223

20 / 223

30 / 223

40 / 223

50 / 223

60 / 223

70 / 223

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=76 accession=GCF_029281585.2

Sleeping i=77 accession=GCF_000151905.2

Sleeping i=78 accession=GCF_008122165.1

80 / 223

Sleeping i=80 accession=GCF_030028045.1

Sleeping i=81 accession=GCF_000001405.40

Sleeping i=81 accession=GCF_000001405.40

Sleeping i=81 accession=GCF_000001405.40

Sleeping i=82 accession=GCF_000002125.1

Sleeping i=83 accession=GCF_000306695.2

Sleeping i=83 accession=GCF_000306695.2

Sleeping i=84 accession=GCF_009914755.1

Sleeping i=84 accession=GCF_009914755.1

Sleeping i=84 accession=GCF_009914755.1

Sleeping i=84 accession=GCF_009914755.1

Sleeping i=84 accession=GCF_009914755.1

Sleeping i=84 accession=GCF_009914755.1

Sleepi

In [38]:
system('echo 1 > /home/frimanav/data/lambda/output/doneinterproscanscheduling', intern = TRUE, wait = TRUE)


character(0)

## Get the list of signatures corresponding to TE proteins

In [4]:
# using TSV files exported from https://www.ebi.ac.uk/interpro/search/text/retroviral/#table
# for search queries "retrovirus", "retroviral", "transpos"
fs=list.files(path = StaticInpDir, pattern = "SearchResults.*tsv", full.names = TRUE)
TEsignaturesToFilterOut=NULL
for (f in fs){
    message(f)
    tmpDF=read.table(f, quote = '"', header=TRUE, sep="\t")
    TEsignaturesToFilterOut=addToDF(TEsignaturesToFilterOut, tmpDF)
}
TEsignaturesToFilterOutDBG=TEsignaturesToFilterOut

# InterPro adds a lot of BS. Double check the key words
keywords=c("trans", "retro")
NoBSbooleanMask=rep(FALSE, nrow(TEsignaturesToFilterOut))
for (keyw in keywords){
    NoBSbooleanMask = NoBSbooleanMask | grepl(pattern = keyw, x = TEsignaturesToFilterOut$Name, ignore.case = TRUE)
    NoBSbooleanMask = NoBSbooleanMask | grepl(pattern = keyw, x = TEsignaturesToFilterOut$Description, ignore.case = TRUE)

}
TEsignaturesToFilterOut=TEsignaturesToFilterOut[NoBSbooleanMask , ]


# InterPro adds a lot of BS. Double check the key words
keywords=c("domain([^s]|$)", "Zinc", "Eukaryot", "KINASE", "FACTOR") # negative KW
NoBSbooleanMask2=rep(FALSE, nrow(TEsignaturesToFilterOut))
for (keyw in keywords){
    NoBSbooleanMask2 = NoBSbooleanMask2 | grepl(pattern = keyw, x = TEsignaturesToFilterOut$Name, ignore.case = TRUE)
    NoBSbooleanMask2 = NoBSbooleanMask2 | grepl(pattern = keyw, x = TEsignaturesToFilterOut$Description, ignore.case = TRUE)

}
TEsignaturesToFilterOut=TEsignaturesToFilterOut[!NoBSbooleanMask2 , ]


NeedBooleanMask= TEsignaturesToFilterOut$Source.Database=="PANTHER"
NeedBooleanMask= NeedBooleanMask | TEsignaturesToFilterOut$Source.Database=="PFAM"
NeedBooleanMask= NeedBooleanMask | grepl(pattern = "polyprotein", x = TEsignaturesToFilterOut$Name, ignore.case = TRUE)
#NeedBooleanMask= NeedBooleanMask | grepl(pattern = "polyprotein", x = TEsignaturesToFilterOut$Description, ignore.case = TRUE)
NeedBooleanMask= NeedBooleanMask | grepl(pattern = "poly-protein", x = TEsignaturesToFilterOut$Name, ignore.case = TRUE)
#NeedBooleanMask= NeedBooleanMask | grepl(pattern = "poly-protein", x = TEsignaturesToFilterOut$Description, ignore.case = TRUE)

TEsignaturesToFilterOut=TEsignaturesToFilterOut[ NeedBooleanMask ,]

/home/frimanav/data/lambda/data//SearchResults-retroviral.tsv

/home/frimanav/data/lambda/data//SearchResultsretrovirus.tsv

/home/frimanav/data/lambda/data//SearchResultstranspos.tsv



In [5]:
nrow(TEsignaturesToFilterOut)

[1] 48

## Reading list of signatures found by interproscan in the genomes

In [6]:
getMaxDigitString=function(inElems){
    #message(typeof(inElems))
    #message(length(inElems))
    maxStringLen=0
    for (inStr in inElems){
        maxStringLen=max(maxStringLen, str_length(inStr))
    }
    return(maxStringLen)
}
getMaxDigitStringVector=function(inStrings){
    outVals=unlist(lapply(str_extract_all(string = inStrings, pattern = "[0-9]+"), FUN = getMaxDigitString))
    return(outVals)
}

In [7]:
TEprotResultsDirs=list.dirs(path = paste0(OutputDir, "filteroutTEgenes"), full.names = TRUE, recursive = FALSE)
TEprotResultsDirs=TEprotResultsDirs[ grepl(pattern = "InterPro\\.G.*", x = TEprotResultsDirs)]

In [8]:
DFtoWrite=NULL
for (i in 1:length(TEprotResultsDirs)){
    TEprotResultsDir = TEprotResultsDirs[i]
    accession=str_remove(string = basename(TEprotResultsDir), pattern = "InterPro.")
    # counting length of digit only sequences in gene names
    queryTxt=sprintf('SELECT DISTINCT gene FROM GTFs WHERE accession="%s" AND type="CDS"',
                     accession)
    genenames=dbGetQuery(mydbRO, queryTxt)[[1]]
    if (any(genenames=="?")){
        queryTxt=sprintf('SELECT DISTINCT Name FROM GTFs WHERE accession="%s" AND type="CDS"',
                     accession)
        genenames=dbGetQuery(mydbRO, queryTxt)[[1]]
    }
    numDigitsInRow=getMaxDigitStringVector(genenames)
    DigitisInRowThreshold = 4*median(numDigitsInRow)
    genenames.ManyDigits=genenames[ numDigitsInRow > DigitisInRowThreshold]
    
    # reading the InterPro outputs
    TSVfiles=list.files(path = TEprotResultsDir, pattern = "protein.*.tsv", full.names = TRUE)
    badGeneSuspects=NULL
    for ( f in TSVfiles ){
        tmpDF=read.table(f, quote = '"', header=FALSE, sep="\t")
        tmpDF=tmpDF[tmpDF$V5 %in% TEsignaturesToFilterOut$Accession , ]
        badGeneSuspects=addToDF(badGeneSuspects, tmpDF)
    
    }
    #print(badGeneSuspects)
    susGeneNames=NULL
    for (proteinID in badGeneSuspects$V1){
        tmpGeneDF=requestGTFbyProteinName(accession, proteinID)
        susGeneNamesTMP=unique(tmpGeneDF$gene)
        if (any(susGeneNames=="?")){
            susGeneNamesTMP=unique(tmpGeneDF$Name)
        }
        susGeneNames=addToVec(susGeneNames, susGeneNamesTMP)
    }
    
    susGeneNamesDigitsRows=getMaxDigitStringVector(susGeneNames)
    namesToRemoveFinal=unique(susGeneNames[ susGeneNamesDigitsRows> DigitisInRowThreshold])
    if (length(namesToRemoveFinal)>0){
        DFtoWriteTMP=data.frame(accession=accession, gene=namesToRemoveFinal)
        DFtoWrite=addToDF(DFtoWrite, DFtoWriteTMP)
        message(accession, ": ", paste0(namesToRemoveFinal, collapse = ", "))
    }else{
        if (i %% 20 == 0){
            message(accession,": ",i," / ", length(TEprotResultsDirs))
        }
    }
}

GCA_000003055.5: LOC100139216, LOC790428, LOC100139382, LOC100138743, LOC100336486, LOC786328, LOC100298055, LOC100295414, LOC781470, LOC100138700, LOC100296151, LOC100140993, LOC100297662, LOC100336014, LOC100299483, LOC100140931, LOC100138733, LOC100336391, LOC100335809

GCA_028583565.1: 20 / 223

GCF_000001895.5: LOC100363193, LOC102549694, LOC102551518, LOC102550650, LOC102553405, LOC102554594, LOC102555474, LOC102551620, LOC102555858, LOC102556213, LOC102555760, LOC102550241, LOC102546456, LOC102547206, LOC102551754, LOC103690189, LOC102550309, LOC102550582, LOC102546424, LOC103690834, LOC102555330, LOC103694538, LOC102552860, LOC102555832, LOC108349832, LOC108349608, LOC108349754, LOC108349573, LOC108349777, LOC108349732, LOC108349537, LOC108348438, LOC108349748, LOC108349511, LOC108349568, LOC108349495, LOC108349842, LOC108349865, LOC108349459, LOC108350189, LOC102551838, LOC108350002, LOC108349907, LOC108350249, LOC103691841, LOC102551194, LOC102552784, LOC103691918, LOC1083505

In [9]:
DigitisInRowThreshold

[1] 4

In [10]:
length(unique(DFtoWrite$accession))

[1] 170

In [11]:
nrow(DFtoWrite)

[1] 5280

In [12]:
DFtoWrite[ !grepl(pattern = "LOC", x = DFtoWrite$gene),]

accession,gene
<chr>,<chr>


In [13]:
# run once
if ("suspectedTEgenes" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE suspectedTEgenes')
dbWriteTable(mydbRW, "suspectedTEgenes", DFtoWrite, append=FALSE)
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS suspectedTEgenesAccessionInd ON suspectedTEgenes(accession, gene)')
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS suspectedTEgenesGeneInd ON suspectedTEgenes(gene, accession)')


[1] 0

[1] 0

# Index FastA files to estimate number of Ns in the future

In [14]:
swarmlines=NULL
sbatchscript="~/data/lambda/notebooks/Multigene.Step3.IndexFasta.sh"

for (accession in unique(GenomesDF$accession)){
    # find genomic fasta file
    accessionDir=paste0(StaticInpDir,accession,"/")
    filepattern=paste0(accession,".*_genomic.fna$")
    fs=list.files(accessionDir, pattern = filepattern, full.names = TRUE)
    if (length(fs)!=1){
        message("Problem with ", accession, " Not a single ref genome file")
        break
    }
    swarmlines=addToVec(swarmlines, paste(sbatchscript, fs))
}


In [15]:
length(swarmlines)

[1] 223

In [16]:

swarmparams=paste0("#SWARM -t 8 -g 20 --time=2:00:00 --gres=lscratch:20 --logdir ", path.expand(OutputDir),"/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"/runningIndexingFastAfiles.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    message("Run from bash: ", fname)
    swarmFiles=addToVec(swarmFiles, fname)
    processed=upperlimit
    i=i+1
}

1 223

Run from bash: ~/data/lambda/output//runningIndexingFastAfiles.0-2026-01-15_09.36.44.820386.swarm

